# **Notebook entregable**

Incluye:

- Exploración y limpieza de datos.
- Tratamiento de variables financieras y textual
- Ingeniería de características
- Clasificación de gastos
- Análisis financiero
- Entrenamiento y evaluación de modelos
- Métricas de desempeño
- Serialización de modelos (.pkl o .joblib)


In [46]:
# ============================================
# Categorías oficiales de consumo del sistema
# ============================================

CATEGORIAS_CONSUMO = [
    "Vivienda",
    "Alimentación",
    "Transporte",
    "Salud",
    "Educación",
    "Entretenimiento y ocio",
    "Suscripciones digitales",
    "Compras personales",
    "Viajes y vacaciones",
    "Otros"
]

# Implementación de motor financiero

## 1.- Recepción de archivo JSON de prueba ya clasificado manualmente 

In [95]:
import json
import random
from pathlib import Path

def cargar_datos_clasificados(carpeta="datos"):
    """
    Selecciona aleatoriamente uno de los archivos JSON clasificados,
    lo convierte a un diccionario y lo devuelve.
    """

    carpeta_json = Path("perfiles_financieros_pruebas_clasificados_para_motor_financiero")

    numero = random.randint(1, 15)

    nombre_archivo = f"{numero:02d}_clasificado_manualmente.json"

    ruta_archivo = carpeta_json / nombre_archivo

    with open(ruta_archivo, "r", encoding="utf-8") as archivo:
        datos = json.load(archivo)

    print(f"Archivo seleccionado: {nombre_archivo}")

    return datos


datos_clasificados = cargar_datos_clasificados()
print (datos_clasificados)

Archivo seleccionado: 14_clasificado_manualmente.json
{'usuario': {'nombre': 'Nicolás Bravo'}, 'periodo': {'inicio': '2026-07-01', 'fin': '2026-07-31'}, 'ingresos': [{'fecha': '2026-07-01', 'descripcion': 'Diseño de sitio web', 'monto': 14000}, {'fecha': '2026-07-02', 'descripcion': 'Proyecto de identidad visual', 'monto': 9500}, {'fecha': '2026-07-03', 'descripcion': 'Consultoría de marketing', 'monto': 7000}, {'fecha': '2026-07-04', 'descripcion': 'Venta de plantillas digitales', 'monto': 3200}], 'transacciones': [{'fecha': '2026-07-02', 'descripcion': 'Renta departamento', 'monto': 7500, 'forma_pago': 'Transferencia bancaria', 'clasificacion': ['Consumo'], 'categoria': 'Vivienda', 'grupo': 'Esencial'}, {'fecha': '2026-07-04', 'descripcion': 'Internet Totalplay', 'monto': 699, 'forma_pago': 'Tarjeta de crédito', 'tasa_de_interes_de_la_tarjeta': 58, 'clasificacion': ['Consumo', 'Pago de deuda'], 'categoria': 'Otros', 'grupo': 'Esencial'}, {'fecha': '2026-07-05', 'descripcion': 'Adobe 

## 2.- Generación de varibles globales

In [97]:
def calcular_ingreso_mensual(datos_clasificados):
    """
    Calcula el ingreso mensual del usuario.

    Parámetros:
        datos_clasificados (dict): Diccionario con la información del usuario.

    Retorna:
        float: Suma de todos los ingresos registrados durante el periodo.
    """

    ingreso_mensual = 0.0

    for ingreso in datos_clasificados["ingresos"]:
        ingreso_mensual += ingreso["monto"]

    return ingreso_mensual

ingreso_mensual = calcular_ingreso_mensual(datos_clasificados)

print(f"Ingreso mensual: ${ingreso_mensual:,.2f}")

Ingreso mensual: $33,700.00


In [98]:
def calcular_consumo_total_mensual(datos_clasificados):
    """
    Calcula el consumo total mensual del usuario.

    Se consideran únicamente las transacciones cuya clasificación
    contenga exclusivamente la etiqueta "Consumo".

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Consumo total mensual.
    """

    consumo_total_mensual = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if (
            "Consumo" in clasificacion
            and len(clasificacion) == 1
        ):
            consumo_total_mensual += transaccion["monto"]

    return consumo_total_mensual

consumo_total_mensual = calcular_consumo_total_mensual(datos_clasificados)

print(f"Consumo total mensual: ${consumo_total_mensual:,.2f}")

Consumo total mensual: $12,800.00


In [99]:
def calcular_pago_mensual_de_deudas(datos_clasificados):
    """
    Calcula el pago mensual de deudas del usuario.

    Se consideran todas las transacciones cuya clasificación
    incluya la etiqueta "Pago de deuda", independientemente
    de si también pertenecen a otra clasificación.

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Pago mensual de deudas.
    """

    pago_mensual_de_deudas = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if "Pago de deuda" in clasificacion:
            pago_mensual_de_deudas += transaccion["monto"]

    return pago_mensual_de_deudas

pago_mensual_de_deudas = calcular_pago_mensual_de_deudas(datos_clasificados)

print(f"Pago mensual de deudas: ${pago_mensual_de_deudas:,.2f}")

Pago mensual de deudas: $9,914.00


In [100]:
def calcular_ahorro_e_inversion_total(datos_clasificados):
    """
    Calcula el ahorro e inversión total del usuario.

    Se consideran todas las transacciones cuya clasificación
    incluya la etiqueta "Ahorro e inversión".

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Ahorro e inversión total.
    """

    ahorro_e_inversion_total = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if "Ahorro e inversión" in clasificacion:
            ahorro_e_inversion_total += transaccion["monto"]

    return ahorro_e_inversion_total

ahorro_e_inversion_total = calcular_ahorro_e_inversion_total(datos_clasificados)

print(f"Ahorro e inversión total: ${ahorro_e_inversion_total:,.2f}")

Ahorro e inversión total: $6,000.00


In [102]:
def calcular_egreso_total(consumo_total_mensual, pago_mensual_de_deudas):
    """
    Calcula el egreso total del usuario.

    El egreso total corresponde a la suma del consumo total mensual
    y el pago mensual de deudas.

    Parámetros:
        consumo_total_mensual (float): Consumo total mensual del usuario.
        pago_mensual_de_deudas (float): Pago mensual de deudas del usuario.

    Retorna:
        float: Egreso total del usuario.
    """

    egreso_total = consumo_total_mensual + pago_mensual_de_deudas

    return egreso_total

egreso_total = calcular_egreso_total(
    consumo_total_mensual,
    pago_mensual_de_deudas
)

print(f"Egreso total: ${egreso_total:,.2f}")

Egreso total: $22,714.00


In [103]:
def calcular_balance_mensual(ingreso_mensual, egreso_total):
    """
    Calcula el balance mensual del usuario.

    El balance mensual corresponde a la diferencia entre el ingreso
    mensual y el egreso total del periodo.

    Parámetros:
        ingreso_mensual (float): Ingreso mensual del usuario.
        egreso_total (float): Egreso total del usuario.

    Retorna:
        float: Balance mensual del usuario.
    """

    balance_mensual = ingreso_mensual - egreso_total

    return balance_mensual

balance_mensual = calcular_balance_mensual(
    ingreso_mensual,
    egreso_total
)

print(f"Balance mensual: ${balance_mensual:,.2f}")

Balance mensual: $10,986.00


In [104]:
def calcular_consumo_total_por_categoria(datos_clasificados):
    """
    Calcula el consumo total por categoría del usuario.

    Se consideran todas las transacciones cuya clasificación
    incluya la etiqueta "Consumo", independientemente de que
    también estén clasificadas como "Pago de deuda".

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Consumo total por categoría.
    """

    consumo_total_por_categoria = 0.0

    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if "Consumo" in clasificacion:
            consumo_total_por_categoria += transaccion["monto"]

    return consumo_total_por_categoria

consumo_total_por_categoria = calcular_consumo_total_por_categoria(datos_clasificados)

print(f"Consumo total por categoría: ${consumo_total_por_categoria:,.2f}")

Consumo total por categoría: $19,514.00


In [105]:
def calcular_gasto_por_categoria(datos_clasificados):
    """
    Calcula el gasto acumulado por categoría de consumo.

    Se consideran todas las transacciones cuya clasificación
    incluya la etiqueta "Consumo", independientemente de que
    también estén clasificadas como "Pago de deuda".

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        dict: Diccionario con el gasto acumulado por cada categoría.
    """

    # Inicializar todas las categorías en 0.0
    gasto_por_categoria = {
        categoria: 0.0
        for categoria in CATEGORIAS_CONSUMO
    }

    # Recorrer las transacciones
    for transaccion in datos_clasificados["transacciones"]:

        clasificacion = transaccion["clasificacion"]

        if "Consumo" in clasificacion:

            categoria = transaccion["categoria"]

            gasto_por_categoria[categoria] += transaccion["monto"]

    return gasto_por_categoria

gasto_por_categoria = calcular_gasto_por_categoria(datos_clasificados)

print("Gasto por categoría:")

for categoria, monto in gasto_por_categoria.items():
    print(f"{categoria}: ${monto:,.2f}")


Gasto por categoría:
Vivienda: $7,500.00
Alimentación: $2,400.00
Transporte: $1,300.00
Salud: $1,600.00
Educación: $0.00
Entretenimiento y ocio: $780.00
Suscripciones digitales: $0.00
Compras personales: $0.00
Viajes y vacaciones: $0.00
Otros: $5,934.00


In [106]:
def calcular_distribucion_porcentual_gasto(
    gasto_por_categoria,
    consumo_total_por_categoria
):
    """
    Calcula la distribución porcentual del gasto por categoría.

    La distribución porcentual representa el porcentaje que ocupa
    cada categoría respecto al consumo total por categoría.

    Parámetros:
        gasto_por_categoria (dict): Gasto acumulado por categoría.
        consumo_total_por_categoria (float): Consumo total por categoría.

    Retorna:
        dict: Distribución porcentual del gasto por categoría.
    """

    # Inicializar todas las categorías en 0.0
    distribucion_porcentual_gasto = {
        categoria: 0.0
        for categoria in CATEGORIAS_CONSUMO
    }

    # Evitar división entre cero
    if consumo_total_por_categoria == 0.0:
        return distribucion_porcentual_gasto

    # Calcular la distribución porcentual
    for categoria in CATEGORIAS_CONSUMO:

        distribucion_porcentual_gasto[categoria] = (
            gasto_por_categoria[categoria]
            / consumo_total_por_categoria
        ) * 100

    return distribucion_porcentual_gasto

distribucion_porcentual_gasto = calcular_distribucion_porcentual_gasto(
    gasto_por_categoria,
    consumo_total_por_categoria
)

print("Distribución porcentual del gasto:")

for categoria, porcentaje in distribucion_porcentual_gasto.items():
    print(f"{categoria}: {porcentaje:.2f}%")

Distribución porcentual del gasto:
Vivienda: 38.43%
Alimentación: 12.30%
Transporte: 6.66%
Salud: 8.20%
Educación: 0.00%
Entretenimiento y ocio: 4.00%
Suscripciones digitales: 0.00%
Compras personales: 0.00%
Viajes y vacaciones: 0.00%
Otros: 30.41%


In [107]:
def calcular_tasa_interes_promedio_ponderada(datos_clasificados):
    """
    Calcula la tasa de interés promedio ponderada de las compras
    realizadas con tarjeta de crédito.

    El promedio se calcula utilizando el monto de cada transacción
    como peso.

    Parámetros:
        datos_clasificados (dict): Diccionario con las transacciones clasificadas.

    Retorna:
        float: Tasa de interés promedio ponderada.
    """

    monto_total_credito = 0.0
    suma_ponderada = 0.0

    # Recorrer las transacciones
    for transaccion in datos_clasificados["transacciones"]:

        if transaccion["forma_pago"] == "Tarjeta de crédito":

            monto = transaccion["monto"]
            tasa = transaccion["tasa_de_interes_de_la_tarjeta"]

            monto_total_credito += monto
            suma_ponderada += monto * tasa

    # Evitar división entre cero
    if monto_total_credito == 0.0:
        return 0.0

    tasa_interes_promedio_ponderada = (
        suma_ponderada / monto_total_credito
    )

    return tasa_interes_promedio_ponderada

tasa_interes_promedio_ponderada = calcular_tasa_interes_promedio_ponderada(
    datos_clasificados
)

print(
    f"Tasa de interés promedio ponderada: "
    f"{tasa_interes_promedio_ponderada:.2f}%"
)

Tasa de interés promedio ponderada: 58.00%
